<a href="https://colab.research.google.com/github/Sakshamrawat12/Customer-Churn-Analysis/blob/main/customer_churn_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# 1. GENERATE REALISTIC CHURN DATA LOCALLY
# ==========================================
print("⚡ Generating structured Customer Churn dataset locally...")

np.random.seed(42)
n_customers = 5000

# Generating continuous behavior features
tenure_months = np.random.randint(1, 72, size=n_customers)          # How long they've stayed
monthly_charges = np.random.uniform(20.0, 120.0, size=n_customers)   # Monthly bill
support_tickets = np.random.poisson(lam=1.5, size=n_customers)       # Tech issues raised

# Generating categorical contract type features
contract_choices = ['Month-to-month', 'One year', 'Two year']
contract_types = np.random.choice(contract_choices, size=n_customers, p=[0.5, 0.25, 0.25])

# Assemble into a base DataFrame
df = pd.DataFrame({
    'TenureMonths': tenure_months,
    'MonthlyCharges': monthly_charges,
    'SupportTickets': support_tickets,
    'ContractType': contract_types
})

# Calculate a realistic probability to determine if a customer leaves (Churn = 1)
# Short tenure, high monthly bills, frequent support tickets, and month-to-month contracts drive churn up
churn_log_odds = (
    -0.05 * df['TenureMonths'] +
    0.02 * df['MonthlyCharges'] +
    0.6 * df['SupportTickets'] +
    1.5 * (df['ContractType'] == 'Month-to-month') - 2.5
)
churn_prob = 1 / (1 + np.exp(-churn_log_odds))
df['Churn'] = np.random.binomial(n=1, p=churn_prob)

print("\n--- Dataset Preview ---")
print(df.head())

print("\n--- Churn Rate Distribution ---")
n_stayed = df['Churn'].value_counts()[0]
n_churned = df['Churn'].value_counts()[1]
print(f"Retained Customers (Class 0): {n_stayed} ({n_stayed/len(df)*100:.2f}%)")
print(f"Churned Customers (Class 1): {n_churned} ({n_churned/len(df)*100:.2f}%)")

# Split features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ==========================================
# 2. DATA PREPROCESSING (ENCODING & SCALING)
# ==========================================
# One-hot encode the categorical 'ContractType' and scale the numeric features
numeric_features = ['TenureMonths', 'MonthlyCharges', 'SupportTickets']
categorical_features = ['ContractType']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
print("\n🤖 Training the Customer Churn Classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_processed, y_train)
print("✅ Training complete!")

# ==========================================
# 4. EVALUATE THE MODEL
# ==========================================
y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("\n--- Model Performance Metrics ---")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"Area Under ROC Curve (ROC-AUC): {roc_auc:.4f}")

# ==========================================
# 5. TEST WITH A CUSTOM AT-RISK CUSTOMER PROFILE
# ==========================================
print("\n🔮 Analyzing an individual customer profile for risk assessment...")

# Creating a profile: Customer has been here only 3 months, paying a high $110/mo bill,
# has called tech support 4 times, and is on a Month-to-month contract.
sample_customer = pd.DataFrame([{
    'TenureMonths': 3,
    'MonthlyCharges': 110.0,
    'SupportTickets': 4,
    'ContractType': 'Month-to-month'
}])

# Preprocess the individual profile sample exactly like training data
sample_customer_processed = preprocessor.transform(sample_customer)

pred = model.predict(sample_customer_processed)[0]
prob = model.predict_proba(sample_customer_processed)[0][1]

status = "⚠️ HIGH RISK OF CHURNING" if pred == 1 else "✅ LOW RISK (RETAINED)"
print(f"Risk Evaluation Result: {status}")
print(f"Calculated Churn Probability: {prob * 100:.2f}%")

⚡ Generating structured Customer Churn dataset locally...

--- Dataset Preview ---
   TenureMonths  MonthlyCharges  SupportTickets    ContractType  Churn
0            52      119.408165               2  Month-to-month      0
1            15       21.107855               2        One year      0
2            61       93.017345               0        One year      0
3            21       77.178932               0        Two year      0
4            24       63.405218               1  Month-to-month      0

--- Churn Rate Distribution ---
Retained Customers (Class 0): 3533 (70.66%)
Churned Customers (Class 1): 1467 (29.34%)

🤖 Training the Customer Churn Classifier...
✅ Training complete!

--- Model Performance Metrics ---
Classification Report:
              precision    recall  f1-score   support

    Retained       0.78      0.82      0.80       707
     Churned       0.51      0.45      0.48       293

    accuracy                           0.71      1000
   macro avg       0.65      